# PANDA - External validation on PBGG-1

This notebook evaluates one or more trained PANDA model families on the external PBGG-1 dataset from Tolkach et al. 2023. It is designed to run on Kaggle and produce:

- majority-vote labels per slide
- inter-pathologist agreement statistics
- per-family external QWK
- ensemble external QWK
- a predictions CSV for `results.md` and the paper

**Attach on Kaggle:**
1. A Kaggle mirror of the PBGG-1 OME-TIFF slides
2. The accompanying grading spreadsheet / CSV for the same 50 slides
3. One or more weight datasets produced by `02b_train_all_folds.ipynb`

**Settings:** GPU T4 ON, Internet ON


In [ ]:
REPO = 'https://github.com/Shashaboii/AIMI_Panda_Challenge.git'
BRANCH = 'main'

IMAGE_SIZE = 512
BATCH_SIZE = 8
N_FOLDS = 5
MAX_SLIDES = None          # set to a small integer for a smoke test
MAJORITY_TIE_BREAK = 'median'   # median, mean, or low
DEFAULT_ORDINAL_MODE = 'threshold'
PBGG_ROOT_OVERRIDE = None
LABELS_CSV_OVERRIDE = None
SLIDE_ID_COLUMN_OVERRIDE = None
GRADE_COLUMNS_OVERRIDE = None   # e.g. ['pathologist_1', ..., 'pathologist_10']

MODEL_FAMILIES = [
    {
        'name': 'b0_smoothl1',
        'weights_dir': '/kaggle/input/panda-effnetb0-5fold-baseline',
        'backbone': 'efficientnet-b0',
        'weight_pattern': 'efficientnetb0_fold{fold}.pth',
        'model_kind': 'baseline',
        'ordinal_mode': 'threshold',
    },
    {
        'name': 'b0_ordinal',
        'weights_dir': '/kaggle/input/panda-effnetb0-ordinal-5fold',
        'backbone': 'efficientnet-b0',
        'weight_pattern': 'efficientnetb0_ordinal_fold{fold}.pth',
        'model_kind': 'baseline',
        'ordinal_mode': 'threshold',
    },
]

OUTPUT_CSV = '/kaggle/working/pbgg_external_predictions.csv'
PBGG_HINTS = ('pbgg', 'tolkach')


In [ ]:
import os
import subprocess

if os.path.exists('/kaggle/working/repo'):
    subprocess.run(['git', '-C', '/kaggle/working/repo', 'fetch', '--all'], check=True)
    subprocess.run(['git', '-C', '/kaggle/working/repo', 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', '/kaggle/working/repo', 'pull'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO, '/kaggle/working/repo'], check=True)
subprocess.run(['git', '-C', '/kaggle/working/repo', 'rev-parse', '--short', 'HEAD'], check=True)


In [ ]:
!pip install -q efficientnet_pytorch tifffile


In [ ]:
import os
import re
import sys
from collections import Counter
from itertools import combinations
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import tifffile
import torch
from torch.utils.data import Dataset

sys.path.insert(0, '/kaggle/working/repo')

from src.dataset import IMAGENET_MEAN, IMAGENET_STD
from src.eval import confusion_matrix_str, qwk
from src.inference import load_model, predict

pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 200)


## Discover PBGG-1 assets

The helpers below try to locate the external slides and grading file automatically. If the printed choice looks wrong, override `PBGG_ROOT_OVERRIDE` or `LABELS_CSV_OVERRIDE` in the config cell and rerun from there.


In [ ]:
def canonicalize_slide_id(value):
    value = str(value).strip()
    value = re.sub(r'\.(ome\.)?tiff?$', '', value, flags=re.IGNORECASE)
    value = re.sub(r'[^a-zA-Z0-9]+', '', value).lower()
    return value

def find_pbgg_assets(root_override=None, labels_override=None, hints=PBGG_HINTS):
    if root_override is not None:
        root = Path(root_override)
        slide_paths = sorted(
            [p for p in root.rglob('*') if p.is_file() and any(s in p.name.lower() for s in ('.ome.tif', '.ome.tiff', '.tif', '.tiff'))]
        )
        csv_paths = sorted(root.rglob('*.csv')) + sorted(root.rglob('*.xlsx'))
        labels_path = Path(labels_override) if labels_override is not None else (csv_paths[0] if csv_paths else None)
        return root, slide_paths, labels_path

    candidates = []
    input_root = Path('/kaggle/input')
    for dataset_dir in sorted(input_root.iterdir()):
        if not dataset_dir.is_dir():
            continue
        tif_paths = [
            p for p in dataset_dir.rglob('*')
            if p.is_file() and any(s in p.name.lower() for s in ('.ome.tif', '.ome.tiff', '.tif', '.tiff'))
        ]
        csv_paths = [p for p in dataset_dir.rglob('*.csv')] + [p for p in dataset_dir.rglob('*.xlsx')]
        if not tif_paths:
            continue
        score = 0
        dataset_name = dataset_dir.name.lower()
        score += sum(h in dataset_name for h in hints) * 10
        score += min(len(tif_paths), 100)
        if csv_paths:
            score += 5
        candidates.append((score, dataset_dir, sorted(tif_paths), sorted(csv_paths)))

    if not candidates:
        raise FileNotFoundError('Could not find any TIFF slides under /kaggle/input')

    candidates.sort(key=lambda item: (item[0], len(item[2])), reverse=True)
    score, root, slide_paths, csv_paths = candidates[0]
    labels_path = Path(labels_override) if labels_override is not None else (csv_paths[0] if csv_paths else None)
    return root, slide_paths, labels_path

PBGG_ROOT, SLIDE_PATHS, LABELS_PATH = find_pbgg_assets(PBGG_ROOT_OVERRIDE, LABELS_CSV_OVERRIDE)

print('PBGG root:', PBGG_ROOT)
print('slides found:', len(SLIDE_PATHS))
print('labels file:', LABELS_PATH)
print('first slides:', [p.name for p in SLIDE_PATHS[:5]])

if LABELS_PATH is None:
    raise FileNotFoundError('No labels CSV/XLSX was found. Attach the grading spreadsheet or set LABELS_CSV_OVERRIDE.')

if MAX_SLIDES is not None:
    SLIDE_PATHS = SLIDE_PATHS[:MAX_SLIDES]
    print('MAX_SLIDES active:', MAX_SLIDES)


In [ ]:
def read_table(path):
    path = Path(path)
    if path.suffix.lower() == '.csv':
        return pd.read_csv(path)
    if path.suffix.lower() in {'.xlsx', '.xls'}:
        return pd.read_excel(path)
    raise ValueError(f'Unsupported label file type: {path}')

def guess_slide_id_column(df):
    preferred = [c for c in df.columns if any(tok in c.lower() for tok in ('slide', 'image', 'file', 'case', 'id'))]
    for col in preferred + list(df.columns):
        series = df[col].dropna().astype(str)
        if len(series) == 0:
            continue
        if series.nunique() >= min(10, len(series)):
            return col
    raise ValueError('Could not identify a slide ID column')

def is_grade_column(series):
    numeric = pd.to_numeric(series, errors='coerce')
    valid = numeric.dropna()
    if len(valid) == 0:
        return False
    if not valid.isin(range(6)).all():
        return False
    return valid.nunique() <= 6

def guess_grade_columns(df, slide_id_col):
    cols = []
    for col in df.columns:
        if col == slide_id_col:
            continue
        if is_grade_column(df[col]):
            cols.append(col)
    if not cols:
        raise ValueError('Could not identify pathologist grading columns automatically')
    filtered = [
        col for col in cols
        if not any(tok in col.lower() for tok in ('majority', 'consensus', 'final', 'vote', 'median', 'mean'))
    ]
    return filtered or cols

def majority_vote(grades, tie_break='median'):
    grades = [int(g) for g in grades if pd.notna(g)]
    if not grades:
        return np.nan, 0, []
    counts = Counter(grades)
    top_count = max(counts.values())
    winners = sorted([grade for grade, count in counts.items() if count == top_count])
    if len(winners) == 1:
        return winners[0], len(grades), winners
    if tie_break == 'median':
        return int(np.median(winners)), len(grades), winners
    if tie_break == 'mean':
        return int(np.rint(np.mean(winners))), len(grades), winners
    if tie_break == 'low':
        return int(min(winners)), len(grades), winners
    raise ValueError(f'Unknown MAJORITY_TIE_BREAK={tie_break!r}')

def pairwise_pathologist_qwks(df, grade_cols):
    rows = []
    for col_a, col_b in combinations(grade_cols, 2):
        pair = df[[col_a, col_b]].dropna()
        if len(pair) == 0:
            continue
        rows.append({
            'pathologist_a': col_a,
            'pathologist_b': col_b,
            'n_slides': len(pair),
            'qwk': qwk(pair[col_a].astype(int).values, pair[col_b].astype(int).values),
        })
    return pd.DataFrame(rows).sort_values('qwk') if rows else pd.DataFrame(columns=['pathologist_a', 'pathologist_b', 'n_slides', 'qwk'])

labels_raw = read_table(LABELS_PATH)
slide_id_col = SLIDE_ID_COLUMN_OVERRIDE or guess_slide_id_column(labels_raw)
grade_cols = GRADE_COLUMNS_OVERRIDE or guess_grade_columns(labels_raw, slide_id_col)

print('labels shape:', labels_raw.shape)
print('slide ID column:', slide_id_col)
print('grade columns:', grade_cols)
display(labels_raw.head())


In [ ]:
slide_df = pd.DataFrame({
    'slide_path': [str(p) for p in SLIDE_PATHS],
})
slide_df['slide_name'] = slide_df.slide_path.map(lambda p: Path(p).name)
slide_df['slide_id_key'] = slide_df.slide_name.map(canonicalize_slide_id)

labels_df = labels_raw.copy()
labels_df['slide_id_key'] = labels_df[slide_id_col].map(canonicalize_slide_id)

majority_rows = []
for _, row in labels_df.iterrows():
    grades = [pd.to_numeric(row[col], errors='coerce') for col in grade_cols]
    majority_label, n_votes, winners = majority_vote(grades, tie_break=MAJORITY_TIE_BREAK)
    majority_rows.append({
        'slide_id_key': row['slide_id_key'],
        'slide_id_raw': row[slide_id_col],
        'majority_label': majority_label,
        'n_votes': n_votes,
        'tied_winners': ','.join(map(str, winners)),
        'had_tie': len(winners) > 1,
    })

majority_df = pd.DataFrame(majority_rows).drop_duplicates(subset=['slide_id_key'])
eval_df = majority_df.merge(slide_df, on='slide_id_key', how='inner')

missing_in_slides = majority_df[~majority_df.slide_id_key.isin(slide_df.slide_id_key)]
extra_slides = slide_df[~slide_df.slide_id_key.isin(majority_df.slide_id_key)]

print('matched slides:', len(eval_df))
print('labels without slide match:', len(missing_in_slides))
print('slides without label match:', len(extra_slides))
print('ties in majority vote:', int(eval_df.had_tie.sum()))

if len(eval_df) == 0:
    raise RuntimeError('No slides matched between the PBGG images and grading file')

pathologist_pairwise = pairwise_pathologist_qwks(labels_df, grade_cols)
if len(pathologist_pairwise):
    print('Pairwise pathologist QWK:')
    print(pathologist_pairwise.qwk.describe()[['mean', 'min', 'max']])

pathologist_vs_majority_rows = []
for col in grade_cols:
    pair = labels_df[['slide_id_key', col]].dropna().merge(eval_df[['slide_id_key', 'majority_label']], on='slide_id_key', how='inner')
    pathologist_vs_majority_rows.append({
        'pathologist': col,
        'n_slides': len(pair),
        'qwk_vs_majority': qwk(pair[col].astype(int).values, pair.majority_label.astype(int).values),
    })
pathologist_vs_majority = pd.DataFrame(pathologist_vs_majority_rows).sort_values('qwk_vs_majority')

display(eval_df[['slide_id_raw', 'slide_name', 'majority_label', 'n_votes', 'had_tie', 'tied_winners']].head(10))
display(pathologist_vs_majority)
if len(pathologist_pairwise):
    display(pathologist_pairwise.head())


## Build a lightweight PBGG inference dataset

The baseline model expects one RGB tensor per slide. For external validation we use the smallest available OME-TIFF pyramid level, resize it to `512x512`, and apply the same ImageNet normalization as the training dataset.


In [ ]:
def read_smallest_level_rgb(path):
    path = Path(path)
    with tifffile.TiffFile(path) as tif:
        series = tif.series[0]
        levels = list(getattr(series, 'levels', [])) or [series]
        chosen = min(levels, key=lambda level: np.prod(level.shape[-2:]) if len(level.shape) >= 2 else np.prod(level.shape))
        arr = chosen.asarray()

    arr = np.asarray(arr)
    while arr.ndim > 3:
        arr = arr[0]
    if arr.ndim == 2:
        arr = np.stack([arr] * 3, axis=-1)
    elif arr.ndim == 3 and arr.shape[0] in (1, 3, 4) and arr.shape[-1] not in (1, 3, 4):
        arr = np.moveaxis(arr, 0, -1)

    if arr.ndim != 3:
        raise ValueError(f'Unexpected TIFF array shape for {path}: {arr.shape}')

    if arr.shape[-1] == 1:
        arr = np.repeat(arr, 3, axis=-1)
    if arr.shape[-1] > 3:
        arr = arr[..., :3]

    arr = arr.astype(np.float32)
    if arr.max() > 0:
        arr = arr / arr.max()
    arr = (arr * 255.0).clip(0, 255).astype(np.uint8)
    return arr

class PBGGThumbnailDataset(Dataset):
    def __init__(self, df, image_size=512):
        self.df = df.reset_index(drop=True)
        self.image_size = image_size

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = read_smallest_level_rgb(row.slide_path)
        img = cv2.resize(img, (self.image_size, self.image_size), interpolation=cv2.INTER_AREA)
        img = img.astype(np.float32) / 255.0
        img = (img - IMAGENET_MEAN) / IMAGENET_STD
        img = np.ascontiguousarray(img.transpose(2, 0, 1))
        return torch.from_numpy(img), torch.tensor(0.0, dtype=torch.float32)

pbgg_ds = PBGGThumbnailDataset(eval_df, image_size=IMAGE_SIZE)
print('dataset size:', len(pbgg_ds))


In [ ]:
sample_x, _ = pbgg_ds[0]
print('sample tensor shape:', tuple(sample_x.shape), 'dtype:', sample_x.dtype)
print('sample slide:', eval_df.iloc[0].slide_name)


## Run external validation

Each model family below is averaged across its 5 folds. Then the family-level predictions are averaged again into the final ensemble.


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

def predict_family(dataset, family):
    fold_preds = []
    missing = []
    for fold in range(N_FOLDS):
        weight_path = os.path.join(family['weights_dir'], family['weight_pattern'].format(fold=fold))
        if not os.path.exists(weight_path):
            missing.append(weight_path)
            continue
        model = load_model(
            weight_path,
            backbone=family.get('backbone', 'efficientnet-b0'),
            device=device,
            model_kind=family.get('model_kind', 'baseline'),
        )
        preds = predict(
            model,
            dataset,
            device,
            batch_size=BATCH_SIZE,
            ordinal_mode=family.get('ordinal_mode', DEFAULT_ORDINAL_MODE),
        )
        fold_preds.append(preds)
        del model
        if device.type == 'cuda':
            torch.cuda.empty_cache()

    if missing:
        raise FileNotFoundError('Missing weights for family ' + family['name'] + ': ' + '; '.join(missing))
    if not fold_preds:
        raise RuntimeError(f'No predictions produced for family {family["name"]}')
    return np.mean(np.stack(fold_preds, axis=0), axis=0)

family_predictions = {}
for family in MODEL_FAMILIES:
    print(f'Running family: {family["name"]}')
    preds = predict_family(pbgg_ds, family)
    family_predictions[family['name']] = preds
    print(f'  external QWK vs majority: {qwk(preds, eval_df.majority_label.values):.4f}')

ensemble_preds = np.mean(np.column_stack(list(family_predictions.values())), axis=1)
print(f'Ensemble external QWK vs majority: {qwk(ensemble_preds, eval_df.majority_label.values):.4f}')


In [ ]:
pred_df = eval_df[['slide_id_raw', 'slide_name', 'slide_path', 'majority_label', 'n_votes', 'had_tie', 'tied_winners']].copy()
for family_name, preds in family_predictions.items():
    pred_df[f'pred_{family_name}'] = preds
pred_df['pred_ensemble'] = ensemble_preds
pred_df.to_csv(OUTPUT_CSV, index=False)

summary_rows = []
for family_name, preds in family_predictions.items():
    summary_rows.append({
        'model': family_name,
        'external_qwk_vs_majority': qwk(preds, pred_df.majority_label.values),
    })
summary_rows.append({
    'model': 'ensemble',
    'external_qwk_vs_majority': qwk(pred_df.pred_ensemble.values, pred_df.majority_label.values),
})
summary = pd.DataFrame(summary_rows).sort_values('external_qwk_vs_majority', ascending=False)

print('Predictions CSV:', OUTPUT_CSV)
display(summary)
print()
print('Ensemble confusion matrix (rows = true, cols = predicted):')
print(confusion_matrix_str(pred_df.pred_ensemble.values, pred_df.majority_label.values))


In [ ]:
if len(pathologist_pairwise):
    pairwise_min = float(pathologist_pairwise.qwk.min())
    pairwise_mean = float(pathologist_pairwise.qwk.mean())
    pairwise_max = float(pathologist_pairwise.qwk.max())
    print(f'Pairwise pathologist QWK range: {pairwise_min:.4f} to {pairwise_max:.4f} (mean {pairwise_mean:.4f})')

pathologist_vs_majority_sorted = pathologist_vs_majority.sort_values('qwk_vs_majority', ascending=False)
print('Pathologist vs majority QWK summary:')
print(pathologist_vs_majority_sorted.qwk_vs_majority.describe()[['mean', 'min', 'max']])
display(pathologist_vs_majority_sorted)


## What to log in `results.md`

After the notebook finishes, log:

- which model family won on PBGG-1
- whether the ensemble beat the single best family
- the final external QWK against majority-vote labels
- the observed pathologist agreement range for context

If the predictions look suspicious, inspect the slide/label matching table and the first few TIFF-derived thumbnails before trusting the number.
